# Schmidt decomposition and canonical MPS

The state coefficients will use the same physical order as a tensor product:
`psi.reshape(d0, d1, ...)` has indices $(s_0,s_1,\ldots)$. A site tensor
`A[a,s,b]` has shape $(D_{\rm left},d,D_{\rm right})$. The first and last
bond dimensions are one. Multiplying adjacent tensors sums their common bond
index and leaves the physical indices in that order.

We use the two-qubit state from **Primer on matrix product states**, followed
by the GHZ and W examples and the canonical-form contractions in the
[lecture notes](https://arxiv.org/abs/2512.02706).
Only NumPy is required, with Python 3.10 or later and NumPy 1.26 or later.
All small dense states are kept so that each tensor contraction can be checked
against the original coefficients.


In [1]:
import numpy as np
np.set_printoptions(precision=8, suppress=True)
rng = np.random.default_rng(20261017)
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
zero, one = I2[:, 0], I2[:, 1]
plus, minus = (zero + one)/np.sqrt(2), (zero - one)/np.sqrt(2)
print("NumPy", np.__version__)


NumPy 2.5.3


## A two-site Schmidt decomposition

The book's state is
$$|\psi_2\rangle=\frac{\sqrt3}{2}|0+\rangle+\frac12|1-\rangle.$$
Its coefficient matrix $M_{s_0s_1}$ has the first physical index on rows and
the second on columns. NumPy returns $M=U\operatorname{diag}(\lambda)V^\dagger$.
We call the final matrix `Vh`. The right Schmidt ket has coefficients
`Vh[alpha,:]`, because $M_{st}=\sum_\alpha U_{s\alpha}\lambda_\alpha
(V^\dagger)_{\alpha t}$. This distinction matters for complex states.
The joined bond in the diagram is the Schmidt index $\alpha$.




In [2]:
psi2 = (np.sqrt(3)*np.kron(zero, plus) + np.kron(one, minus))/2
M = psi2.reshape(2, 2)
U, schmidt, Vh = np.linalg.svd(M, full_matrices=False)
assert np.allclose((U*schmidt) @ Vh, M)
assert np.allclose(schmidt, [np.sqrt(3)/2, 0.5])
print("Coefficient matrix:\n", M.real)
print("Schmidt coefficients:", schmidt)


Coefficient matrix:
 [[ 0.61237244  0.61237244]
 [ 0.35355339 -0.35355339]]
Schmidt coefficients: [0.8660254 0.5      ]


Tracing out the right site contracts its ket and bra indices:
$(\rho_L)_{ss'}=\sum_t M_{st}\overline{M_{s't}}$. On the other side,
$(\rho_R)_{tt'}=\sum_s M_{st}\overline{M_{st'}}$, so the array formulas are
$\rho_L=MM^\dagger$ and $\rho_R=M^T\overline M$.
Both density matrices have eigenvalues $\lambda_\alpha^2$.
We use natural logarithms, so the entropy is measured in nats.


In [3]:
rho_left = M @ M.conj().T
rho_right = M.T @ M.conj()
probabilities = schmidt**2
entropy = -np.sum(probabilities*np.log(probabilities))
assert np.allclose(np.linalg.eigvalsh(rho_left), sorted(probabilities))
assert np.allclose(np.linalg.eigvalsh(rho_right), sorted(probabilities))
print("Reduced density matrices:\n", rho_left.real, "\n", rho_right.real)
print(f"Entropy = {entropy:.9f} nats")


Reduced density matrices:
 [[ 0.75 -0.  ]
 [-0.    0.25]] 
 [[0.5  0.25]
 [0.25 0.5 ]]
Entropy = 0.562335145 nats


## Keeping one Schmidt state

Dropping the second term produces an unnormalized vector
$|\widetilde\psi\rangle=(\sqrt3/2)|0+\rangle$. Its squared norm is $3/4$.
After division by that norm's square root, the approximation is $|0+\rangle$.
For a single Schmidt cut, the discarded weight $w=\sum_{\alpha>\chi}
\lambda_\alpha^2$ gives both $\|\psi-\widetilde\psi\|^2=w$ and normalized
fidelity $|\langle\psi|\psi_\chi\rangle|^2=1-w$.




In [4]:
chi = 1
raw = ((U[:, :chi]*schmidt[:chi]) @ Vh[:chi]).reshape(-1)
kept = raw/np.linalg.norm(raw)
discarded_weight = np.sum(schmidt[chi:]**2)
fidelity = abs(np.vdot(psi2, kept))**2
assert np.isclose(np.linalg.norm(psi2-raw)**2, discarded_weight)
assert np.isclose(fidelity, 1-discarded_weight)
assert np.isclose(abs(np.vdot(kept, np.kron(zero, plus))), 1)
print(f"Discarded weight = {discarded_weight:.6f}")
print(f"Normalized fidelity = {fidelity:.6f}")


Discarded weight = 0.250000
Normalized fidelity = 0.750000


The magnetization $Z\otimes I$ measures the changed Schmidt probabilities.
The correlator $X\otimes Z$ exchanges $|0+\rangle$ and $|1-\rangle$ and measures
their coherence. Its off-diagonal contribution disappears in the rank-one state.
For either observable, compare the actual error with
$|\langle O\rangle_\psi-\langle O\rangle_{\psi_\chi}|
\le 2\|O\|\sqrt w$. Here $\|O\|$ is the operator norm.


In [5]:
def dense_expectation(state, operator):
    return np.vdot(state, operator @ state)/np.vdot(state, state)

observables = {"Z on site 0": np.kron(Z, I2), "X tensor Z": np.kron(X, Z)}
expected = [(0.5, 1.0), (np.sqrt(3)/2, 0.0)]
for (name, operator), reference in zip(observables.items(), expected):
    exact = dense_expectation(psi2, operator).real
    reduced = dense_expectation(kept, operator).real
    bound = 2*np.linalg.norm(operator, 2)*np.sqrt(discarded_weight)
    assert np.allclose([exact, reduced], reference)
    assert abs(exact-reduced) <= bound + 1e-12
    print(f"{name:14s}: exact {exact:.8f}, truncated {reduced:.8f}, bound {bound:.8f}")


Z on site 0   : exact 0.50000000, truncated 1.00000000, bound 1.00000000
X tensor Z    : exact 0.86602540, truncated 0.00000000, bound 1.00000000


## Sequential SVD: one physical index at a time

At site $k$, the remaining coefficient array has shape
$(D_{\rm left},d_k,d_{k+1},\ldots)$. Group $(a,s_k)$ into a matrix row and
the remaining physical indices into a column $r$:
$$M_{(a,s_k),r}=\sum_b U_{(a,s_k),b}\lambda_b(V^\dagger)_{br}.$$
Reshape $U$ into `A[a,s,b]`. The new remainder is
$\operatorname{diag}(\lambda)V^\dagger$, with shape $(D_{\rm new},R)$.
Because $U^\dagger U=I$, this tensor is left canonical:
$\sum_{a,s}\overline{A_{asb}}A_{asc}=\delta_{bc}$.
The code removes singular values below a roundoff threshold. It performs no
chosen bond-dimension truncation.




In [6]:
def state_to_mps(state, dimensions, tolerance=1e-13):
    assert state.size == np.prod(dimensions)
    remainder = np.asarray(state).reshape(1, -1)
    tensors, spectra = [], []
    left = 1
    for physical in dimensions[:-1]:
        matrix = remainder.reshape(left*physical, -1)
        u, singular, vh = np.linalg.svd(matrix, full_matrices=False)
        rank = np.count_nonzero(singular > tolerance*singular[0])
        tensors.append(u[:, :rank].reshape(left, physical, rank))
        spectra.append(singular[:rank].copy())
        remainder = singular[:rank, None]*vh[:rank]
        left = rank
    tensors.append(remainder.reshape(left, dimensions[-1], 1))
    return tensors, spectra

def mps_to_state(tensors):
    coefficients = tensors[0][0]
    for tensor in tensors[1:]:
        coefficients = np.tensordot(coefficients, tensor, axes=(-1, 0))
    return coefficients[..., 0].reshape(-1)


For three qubits, the GHZ state carries two equally weighted alternatives,
$|000\rangle$ and $|111\rangle$. In the W state, the excitation lies on one
of three sites. Across the first cut, it lies on the left with probability
$1/3$ and on the right with probability $2/3$. The second cut exchanges these
probabilities. Reconstructing all eight amplitudes checks the index grouping
used in both SVDs.


In [7]:
ghz = np.zeros(8, dtype=complex)
ghz[[0, 7]] = 1/np.sqrt(2)
w = np.zeros(8, dtype=complex)
w[[1, 2, 4]] = 1/np.sqrt(3)
for name, state, expected_squares in [
    ("GHZ", ghz, [0.5, 0.5]), ("W", w, [2/3, 1/3])
]:
    tensors, spectra = state_to_mps(state, [2, 2, 2])
    assert np.allclose(mps_to_state(tensors), state)
    for tensor in tensors[:-1]:
        gram = np.einsum("asb,asc->bc", tensor.conj(), tensor)
        assert np.allclose(gram, np.eye(tensor.shape[2]))
    for singular in spectra:
        assert np.allclose(singular**2, expected_squares)
    print(name, "tensor shapes:", [a.shape for a in tensors])
    print("  Schmidt probabilities:", [s**2 for s in spectra])


GHZ tensor shapes: [(1, 2, 2), (2, 2, 2), (2, 2, 1)]
  Schmidt probabilities: [array([0.5, 0.5]), array([0.5, 0.5])]
W tensor shapes: [(1, 2, 2), (2, 2, 2), (2, 2, 1)]
  Schmidt probabilities: [array([0.66666667, 0.33333333]), array([0.66666667, 0.33333333])]


## Moving the canonical centre with QR

The sequential SVD leaves the norm and remaining amplitudes in the final
tensor. To move a centre tensor to the right, reshape it as
$M_{(a,s),b}$ and take $M=QR$. Reshape $Q$ back to a left-canonical site,
then absorb $R_{cb}$ into the next tensor:
$$A^{[k+1]'}_{ctd}=\sum_b R_{cb}A^{[k+1]}_{btd}.$$
This is an exact factorization. No singular values are discarded.




In [8]:
def move_centre_right(tensors, site):
    left, physical, right = tensors[site].shape
    q, r = np.linalg.qr(tensors[site].reshape(left*physical, right), mode="reduced")
    tensors[site] = q.reshape(left, physical, q.shape[1])
    tensors[site+1] = np.einsum("cb,bsd->csd", r, tensors[site+1])

def move_centre_left(tensors, site):
    left, physical, right = tensors[site].shape
    matrix = tensors[site].reshape(left, physical*right)
    q, r = np.linalg.qr(matrix.conj().T, mode="reduced")
    tensors[site] = q.conj().T.reshape(q.shape[1], physical, right)
    tensors[site-1] = np.einsum("asb,bc->asc", tensors[site-1], r.conj().T)


Moving left uses the same QR routine on $M^\dagger$, where
$M_{a,(s,b)}=A_{asb}$. If $M^\dagger=QR$, then
$M=R^\dagger Q^\dagger$. The new right-canonical tensor is $Q^\dagger$,
and $R^\dagger$ is absorbed on its left. Its rows obey
$\sum_{s,b}A_{asb}\overline{A_{csb}}=\delta_{ac}$.
The conjugate transpose is required for complex amplitudes.

Start with a normalized complex four-qubit state, move its centre from site 3
to site 0, then back to site 2. Sites 0 and 1 should be left canonical, site 3
right canonical, and the centre should have Frobenius norm one.


In [9]:
complex_state = rng.normal(size=16) + 1j*rng.normal(size=16)
complex_state /= np.linalg.norm(complex_state)
canonical, _ = state_to_mps(complex_state, [2]*4)
for site in range(3, 0, -1):
    move_centre_left(canonical, site)
for site in range(2):
    move_centre_right(canonical, site)
assert np.allclose(mps_to_state(canonical), complex_state)
for tensor in canonical[:2]:
    assert np.allclose(np.einsum("asb,asc->bc", tensor.conj(), tensor),
                       np.eye(tensor.shape[2]))
for tensor in canonical[3:]:
    assert np.allclose(np.einsum("asb,csb->ac", tensor, tensor.conj()),
                       np.eye(tensor.shape[0]))
assert np.isclose(np.linalg.norm(canonical[2]), 1)
print("Centre at site 2, shapes:", [a.shape for a in canonical])
print("Dense reconstruction error:", np.linalg.norm(mps_to_state(canonical)-complex_state))


Centre at site 2, shapes: [(1, 2, 2), (2, 2, 4), (4, 2, 2), (2, 2, 1)]
Dense reconstruction error: 5.29079284814734e-16


## Contracting an observable through the chain

For a product operator $O_0\otimes\cdots\otimes O_{N-1}$, let $E_{ac}$
contain the contraction to the left of a site. Its first index belongs to
the ket and its second to the bra. Adding one site gives
$$E'_{bd}=\sum_{a,c,s,t}E_{ac}A_{asb}\overline{A_{ctd}}(O_k)_{ts}.$$
The operator's row is the bra physical index $t$, and its column is the ket
index $s$. Begin with the scalar boundary $E=(1)$ and end with a scalar.
Identity operators give the state norm. This contraction works in any gauge.




In [10]:
def product_expectation(tensors, operators):
    assert len(tensors) == len(operators)
    environment = np.ones((1, 1), dtype=complex)
    for tensor, operator in zip(tensors, operators):
        environment = np.einsum("ac,asb,ctd,ts->bd", environment,
                                tensor, tensor.conj(), operator, optimize=True)
    return environment[0, 0]

def product_matrix(operators):
    result = np.ones((1, 1), dtype=complex)
    for operator in operators:
        result = np.kron(result, operator)
    return result

ops = [Z, I2, Y, X]
network_value = product_expectation(canonical, ops)
dense_value = dense_expectation(complex_state, product_matrix(ops))
assert np.allclose(network_value, dense_value)
assert np.isclose(product_expectation(canonical, [I2]*4), 1)
print("Network expectation:", network_value)
print("Dense expectation:  ", dense_value)


Network expectation: (-0.29291169275516615+6.938893903907228e-17j)
Dense expectation:   (-0.2929116927551662+0j)


For an operator acting only on the centre site, the canonical contractions
on both sides reduce to identities. The reduced density matrix is then
$\rho_{st}=\sum_{a,b}A_{asb}\overline{A_{atb}}$.
This gives the same result as contracting all four sites. If a gauge change
destroys the surrounding isometries, this local shortcut no longer applies,
while the environment contraction remains valid.


In [11]:
centre = canonical[2]
rho_centre = np.einsum("asb,atb->st", centre, centre.conj())
local_value = np.trace(rho_centre @ Y)
assert np.isclose(np.trace(rho_centre), 1)
assert np.allclose(local_value, product_expectation(canonical, [I2, I2, Y, I2]))
gauged = [a.copy() for a in canonical]
bond_dimension = gauged[1].shape[2]
gauge = np.diag(np.linspace(0.7, 1.4, bond_dimension)).astype(complex)
gauged[1] = np.einsum("asb,bc->asc", gauged[1], gauge)
next_shape = gauged[2].shape
gauged[2] = np.linalg.solve(gauge, gauged[2].reshape(bond_dimension, -1)).reshape(next_shape)
assert np.allclose(mps_to_state(gauged), complex_state)
assert np.allclose(product_expectation(gauged, ops), dense_value)
print("Centre-site <Y>:", local_value)
print("Expectation after a nonunitary bond gauge:", product_expectation(gauged, ops))


Centre-site <Y>: (0.4100853545859191+0j)
Expectation after a nonunitary bond gauge: (-0.2929116927551662+1.3877787807814457e-17j)


## Exercise 1: a relative phase and two correlators

Replace the second Schmidt amplitude by $e^{i\varphi}/2$ and use
$\varphi=0.7$. Predict which reduced-density eigenvalues change. Compute
$\langle X\otimes Z\rangle$ and $\langle Y\otimes Z\rangle$, then compare
with $(\sqrt3/2)\cos\varphi$ and $(\sqrt3/2)\sin\varphi$.
The calculation below supplies the comparison and also checks the complex
right-density-matrix contraction.


In [12]:
phase = 0.7
phase_state = (np.sqrt(3)*np.kron(zero, plus)
               + np.exp(1j*phase)*np.kron(one, minus))/2
phase_matrix = phase_state.reshape(2, 2)
phase_singular = np.linalg.svd(phase_matrix, compute_uv=False)
assert np.allclose(phase_singular, schmidt)
rho_right_phase = np.einsum("st,su->tu", phase_matrix, phase_matrix.conj())
assert np.allclose(rho_right_phase, phase_matrix.T @ phase_matrix.conj())
predicted = np.sqrt(3)/2*np.array([np.cos(phase), np.sin(phase)])
measured = np.array([dense_expectation(phase_state, np.kron(o, Z)).real for o in (X, Y)])
assert np.allclose(measured, predicted)
print("<XZ>, <YZ>:", measured)
print("Schmidt probabilities:", phase_singular**2)


<XZ>, <YZ>: [0.66237276 0.55790888]
Schmidt probabilities: [0.75 0.25]


## Exercise 2: the W-state entropy across an unequal cut

Construct $|W_5\rangle=5^{-1/2}\sum_j|0\cdots1_j\cdots0\rangle$ and cut
after the first two sites. The excitation is on the left with probability
$2/5$ and on the right with probability $3/5$. Compare the SVD probabilities
and entropy with these two numbers. Contrast the result with a five-site
GHZ state, whose entropy is $\log 2$ at every nontrivial cut.


In [13]:
w5 = np.zeros(32, dtype=complex)
w5[[1 << j for j in range(5)]] = 1/np.sqrt(5)
w5_mps, w5_spectra = state_to_mps(w5, [2]*5)
assert np.allclose(mps_to_state(w5_mps), w5)
cut_probabilities = w5_spectra[1]**2
assert np.allclose(cut_probabilities, [3/5, 2/5])
w_entropy = -np.sum(cut_probabilities*np.log(cut_probabilities))
print("W5 cut probabilities:", cut_probabilities)
print(f"W5 entropy {w_entropy:.9f}, GHZ5 entropy {np.log(2):.9f}")


W5 cut probabilities: [0.6 0.4]
W5 entropy 0.673011667, GHZ5 entropy 0.693147181


## Exercise 3: saturating the observable bound

For the rank-one approximation to $|\psi_2\rangle$, form
$\Delta\rho=|\psi_2\rangle\langle\psi_2|-|\psi_\chi\rangle\langle\psi_\chi|$.
Choose $O$ to have eigenvalue ${+1}$ on its positive eigenspace and ${-1}$ on
its negative eigenspace. Set its eigenvalue to zero on the nullspace.
Then $\|O\|=1$ and $\operatorname{Tr}(O\Delta\rho)=\|\Delta\rho\|_1$.
Check that this observable attains $2\sqrt w=1$, even though the earlier
Pauli observables had smaller errors.


In [14]:
difference = np.outer(psi2, psi2.conj()) - np.outer(kept, kept.conj())
values, vectors = np.linalg.eigh(difference)
signs = np.where(abs(values) < 1e-12, 0, np.sign(values))
optimal_observable = (vectors*signs) @ vectors.conj().T
error = np.trace(optimal_observable @ difference).real
assert np.isclose(np.linalg.norm(optimal_observable, 2), 1)
assert np.isclose(error, np.sum(abs(values)))
assert np.isclose(error, 2*np.sqrt(discarded_weight))
print(f"Maximal observable error at operator norm one: {error:.9f}")


Maximal observable error at operator norm one: 1.000000000
